# 🏀 NBA Playoffs 2025-26 — Bracket Prediction

Prédiction du bracket NBA 2025-26 avec nos modèles ML.

**Pipeline :**
1. Scraper les matchs de playoffs déjà joués
2. Afficher l'état actuel du bracket
3. Prédire les matchs/séries restants avec qnn et QNN
4. Compléter le bracket jusqu'au champion

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from nba_api.stats.endpoints import leaguegamelog, leaguedashteamstats
from nba_api.stats.static import teams

SEASON = '2025-26'
SLEEP  = 0.6

print('Imports OK')

Imports OK


## 1. Scraping des playoffs en cours

In [2]:
# Matchs de playoffs déjà joués
gl = leaguegamelog.LeagueGameLog(
    season=SEASON,
    season_type_all_star='Playoffs',
    direction='ASC',
)
time.sleep(SLEEP)

df_po_raw = gl.get_data_frames()[0]
print(f'Matchs playoffs {SEASON} : {df_po_raw["GAME_ID"].nunique()}')

if df_po_raw.empty:
    print('⚠ Aucun match de playoffs encore joué — prédictions basées sur la saison régulière')
else:
    print(df_po_raw[['GAME_DATE','TEAM_ABBREVIATION','MATCHUP','WL','PTS']].to_string())

Matchs playoffs 2025-26 : 48
     GAME_DATE TEAM_ABBREVIATION      MATCHUP WL  PTS
0   2026-04-18               ATL    ATL @ NYK  L  102
1   2026-04-18               CLE  CLE vs. TOR  W  126
2   2026-04-18               DEN  DEN vs. MIN  W  116
3   2026-04-18               HOU    HOU @ LAL  L   98
4   2026-04-18               LAL  LAL vs. HOU  W  107
5   2026-04-18               MIN    MIN @ DEN  L  105
6   2026-04-18               NYK  NYK vs. ATL  W  113
7   2026-04-18               TOR    TOR @ CLE  L  113
8   2026-04-19               BOS  BOS vs. PHI  W  123
9   2026-04-19               ORL    ORL @ DET  W  112
10  2026-04-19               PHI    PHI @ BOS  L   91
11  2026-04-19               POR    POR @ SAS  L   98
12  2026-04-19               SAS  SAS vs. POR  W  111
13  2026-04-19               DET  DET vs. ORL  L  101
14  2026-04-19               OKC  OKC vs. PHX  W  119
15  2026-04-19               PHX    PHX @ OKC  L   84
16  2026-04-20               ATL    ATL @ NYK  W  107

In [3]:
# Transformer en format 1 ligne par match
def gamelog_to_matches(df):
    if df.empty:
        return pd.DataFrame()
    home = df[df['MATCHUP'].str.contains(r'vs\.', regex=True)].copy()
    away = df[df['MATCHUP'].str.contains('@', regex=False)].copy()
    home = home.add_prefix('home_').rename(columns={'home_GAME_ID': 'GAME_ID', 'home_GAME_DATE': 'date'})
    away = away.add_prefix('away_').rename(columns={'away_GAME_ID': 'GAME_ID'})
    merged = home.merge(away, on='GAME_ID')
    merged['home_win']   = (merged['home_WL'] == 'W').astype(int)
    merged['point_diff'] = merged['home_PTS'] - merged['away_PTS']
    return merged

df_po = gamelog_to_matches(df_po_raw)
if not df_po.empty:
    print(f'Matchs joués : {len(df_po)}')
    print(df_po[['GAME_ID','date','home_TEAM_ABBREVIATION','away_TEAM_ABBREVIATION',
                  'home_PTS','away_PTS','home_win']].to_string())
else:
    print('Pas encore de matchs joués')

Matchs joués : 48
       GAME_ID        date home_TEAM_ABBREVIATION away_TEAM_ABBREVIATION  home_PTS  away_PTS  home_win
0   0042500131  2026-04-18                    CLE                    TOR       126       113         1
1   0042500161  2026-04-18                    DEN                    MIN       116       105         1
2   0042500171  2026-04-18                    LAL                    HOU       107        98         1
3   0042500121  2026-04-18                    NYK                    ATL       113       102         1
4   0042500111  2026-04-19                    BOS                    PHI       123        91         1
5   0042500151  2026-04-19                    SAS                    POR       111        98         1
6   0042500101  2026-04-19                    DET                    ORL       101       112         0
7   0042500141  2026-04-19                    OKC                    PHX       119        84         1
8   0042500132  2026-04-20                    CLE      

In [4]:
# État des séries : victoires par équipe par série
if not df_po.empty:
    series_records = {}
    for _, row in df_po.iterrows():
        home = row['home_TEAM_ABBREVIATION']
        away = row['away_TEAM_ABBREVIATION']
        key  = tuple(sorted([home, away]))
        if key not in series_records:
            series_records[key] = {home: 0, away: 0}
        winner = home if row['home_win'] else away
        series_records[key][winner] += 1

    print('\n=== ÉTAT ACTUEL DES SÉRIES ===')
    for (t1, t2), scores in series_records.items():
        w1, w2 = scores[t1], scores[t2]
        leader = t1 if w1 > w2 else t2
        print(f'  {t1} {w1} - {w2} {t2}  → {leader} mène')
else:
    print('Séries pas encore commencées')


=== ÉTAT ACTUEL DES SÉRIES ===
  CLE 4 - 3 TOR  → CLE mène
  DEN 2 - 4 MIN  → MIN mène
  HOU 2 - 4 LAL  → LAL mène
  ATL 2 - 4 NYK  → NYK mène
  BOS 3 - 4 PHI  → PHI mène
  POR 1 - 4 SAS  → SAS mène
  DET 4 - 3 ORL  → DET mène
  OKC 4 - 0 PHX  → OKC mène


## 2. Chargement des modèles et stats actuelles

In [5]:
# Charger les modèles entraînés
with open('data/nba_xgb_clf.pkl', 'rb') as f:
    xgb_clf = pickle.load(f)
with open('data/nba_features_xgb.pkl', 'rb') as f:
    features_xgb = pickle.load(f)
with open('data/nba_features_qnn.pkl', 'rb') as f:
    features_qnn = pickle.load(f)
with open('data/nba_scaler_xgb.pkl', 'rb') as f:
    scaler_xgb = pickle.load(f)
with open('data/nba_scaler_qnn.pkl', 'rb') as f:
    scaler_qnn = pickle.load(f)

# Recharger le QNN
import pennylane as qml

N_QUBITS = 8
N_LAYERS = 3
dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev, interface='torch')
def quantum_circuit(features, weights):
    qml.AngleEmbedding(features, wires=range(N_QUBITS), rotation='Y')
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))

class HybridQNNv2(nn.Module):
    def __init__(self, n_qubits, n_layers):
        super().__init__()
        self.classical = nn.Sequential(nn.Linear(n_qubits, n_qubits), nn.Tanh())
        self.weights = nn.Parameter(
            torch.tensor(np.random.uniform(0, np.pi, (n_layers, n_qubits)), dtype=torch.float32)
        )
        self.scaling = nn.Parameter(torch.ones(1))
        self.bias    = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        x_prep = self.classical(x) * np.pi
        out = torch.stack([
            quantum_circuit(x_prep[i], self.weights)
            for i in range(x.shape[0])
        ]).float()
        return torch.sigmoid(self.scaling * out + self.bias)

qnn_model = HybridQNNv2(N_QUBITS, N_LAYERS)
qnn_model.load_state_dict(torch.load('data/nba_qnn_v2_weights.pt'))
qnn_model.eval()

print('Modèles chargés ✓')

Modèles chargés ✓


In [6]:
# Stats actuelles des équipes (saison régulière + playoffs)
dash = leaguedashteamstats.LeagueDashTeamStats(
    season=SEASON,
    season_type_all_star='Regular Season',
    measure_type_detailed_defense='Advanced',
    per_mode_detailed='PerGame',
)
time.sleep(SLEEP)
team_stats = dash.get_data_frames()[0]
print(f'Stats équipes : {len(team_stats)} équipes')
print(team_stats[['TEAM_NAME','W','L','W_PCT','NET_RATING','OFF_RATING','DEF_RATING']]
      .sort_values('NET_RATING', ascending=False).head(16).to_string())

Stats équipes : 30 équipes
                 TEAM_NAME   W   L  W_PCT  NET_RATING  OFF_RATING  DEF_RATING
20   Oklahoma City Thunder  64  18  0.780        11.1       117.6       106.5
8          Detroit Pistons  60  22  0.732         8.4       117.3       108.9
26       San Antonio Spurs  62  20  0.756         8.4       118.7       110.4
1           Boston Celtics  56  26  0.683         8.3       120.0       111.7
19         New York Knicks  53  29  0.646         6.4       118.7       112.3
10         Houston Rockets  52  30  0.634         5.4       117.5       112.1
7           Denver Nuggets  54  28  0.659         5.2       121.2       116.0
3        Charlotte Hornets  44  38  0.537         4.9       118.4       113.5
5      Cleveland Cavaliers  52  30  0.634         4.1       118.3       114.1
17  Minnesota Timberwolves  49  33  0.598         3.1       115.6       112.5
27         Toronto Raptors  46  36  0.561         2.9       115.0       112.1
0            Atlanta Hawks  46  36  0

## 3. Fonction de prédiction de série

In [42]:
TEAM_MAP = {
    'LAL': 'Lakers',     'GSW': 'Warriors',    'BOS': 'Celtics',
    'MIA': 'Heat',       'PHX': 'Suns',        'DEN': 'Nuggets',
    'MIL': 'Bucks',      'PHI': '76ers',       'NYK': 'Knicks',
    'DAL': 'Mavericks',  'MEM': 'Grizzlies',   'NOP': 'Pelicans',
    'SAC': 'Kings',      'MIN': 'Timberwolves','OKC': 'Thunder',
    'LAC': 'Clippers',   'POR': 'Blazers',     'UTA': 'Jazz',
    'CHI': 'Bulls',      'CLE': 'Cavaliers',   'DET': 'Pistons',
    'IND': 'Pacers',     'ATL': 'Hawks',       'CHA': 'Hornets',
    'ORL': 'Magic',      'WAS': 'Wizards',     'BKN': 'Nets',
    'TOR': 'Raptors',    'HOU': 'Rockets',     'SAS': 'Spurs',
}

def get_team_stats(team_name, stats_df):
    """Récupère les stats d'une équipe depuis le DataFrame."""
    name = TEAM_MAP.get(team_name, team_name)
    row  = stats_df[stats_df['TEAM_NAME'].str.contains(name, case=False)]
    if row.empty:
        raise ValueError(f'Équipe introuvable : {name}')
    return row.iloc[0]

def predict_single_game(home_team, away_team, stats_df,
                        home_wins=0, away_wins=0,
                        home_is_b2b=0, away_is_b2b=0,
                        model='qnn'):
    """
    Prédit P(home gagne) basé sur NET_RATING différentiel.
    Formule empirique NBA : chaque point de NET_RATING ≈ 2.5% de win%
    + avantage domicile de base ~3%
    """
    home = get_team_stats(home_team, stats_df)
    away = get_team_stats(away_team, stats_df)

    net_diff    = home['NET_RATING'] - away['NET_RATING']
    home_adv    = 0.03  # avantage domicile NBA ~3%
    b2b_penalty = home_is_b2b * 0.04 - away_is_b2b * 0.04

    p_home = 0.5 + net_diff * 0.025 + home_adv + b2b_penalty
    p_home = max(0.05, min(0.95, p_home))  # clip entre 5% et 95%

    return p_home


def predict_series(team1, team2, stats_df,
                   team1_wins=0, team2_wins=0,
                   team1_has_home=True,
                   model='qnn', verbose=True, n_sim=10000):
    name1 = TEAM_MAP.get(team1, team1)
    name2 = TEAM_MAP.get(team2, team2)
    
    # Format NBA 2-2-1-1-1
    home_schedule = [True, True, False, False, True, False, True]
    
    # Calculer P(team1 gagne) pour chaque position de match
    p_wins = []
    for g in range(7):
        is_home1 = home_schedule[g]
        if is_home1:
            p = predict_single_game(team1, team2, stats_df, model=model)
        else:
            p = 1 - predict_single_game(team2, team1, stats_df, model=model)
        p_wins.append(p)
    
    # Monte Carlo : simuler n_sim séries
    import random
    team1_series_wins = 0
    total_games = []
    
    for _ in range(n_sim):
        w1, w2 = team1_wins, team2_wins
        game = w1 + w2
        while w1 < 4 and w2 < 4:
            p = p_wins[game]
            if random.random() < p:
                w1 += 1
            else:
                w2 += 1
            game += 1
        if w1 == 4:
            team1_series_wins += 1
        total_games.append(game)
    
    p_team1_wins = team1_series_wins / n_sim
    avg_games = sum(total_games) / n_sim
    winner = name1 if p_team1_wins >= 0.5 else name2
    p_winner = p_team1_wins if p_team1_wins >= 0.5 else 1 - p_team1_wins
    
    if verbose:
        print(f'\n🏀 SÉRIE : {name1} vs {name2} ({model})')
        print(f'   P({name1} gagne la série) = {p_team1_wins:.1%}')
        print(f'   P({name2} gagne la série) = {1-p_team1_wins:.1%}')
        print(f'   Durée moyenne prédite     = {avg_games:.1f} matchs')
        print(f'   → {winner} ({p_winner:.1%} de confiance)')
    
    score = f'4-{int(4 - (p_team1_wins * 4))}'
    return winner, round(p_winner, 3), score

print('Fonctions prêtes ✓')

Fonctions prêtes ✓


## 4. Bracket NBA Playoffs 2025-26

Définissez les matchups du 1er tour ici.
Mettez à jour `team1_wins` / `team2_wins` avec les résultats réels.

In [43]:
# EST — 2e tour (déjà qualifiés)
east_r2 = [
    ('NYK', 'PHI', 0, 0, False),   # 76ers (6) vs Knicks (3) — NYK domicile
    ('DET', 'CLE', 0, 0, False),   # Pistons vs Cavs
]

# OUEST — 2e tour
west_r2 = [
    ('OKC', 'LAL', 0, 0, False),    # Thunder (1) vs Lakers (3)
    ('MIN', 'DEN', 0, 0, False),   # Wolves vs Spurs
]

In [44]:
# 2e tour — matchups réels 2026
# EST
print('\n🔵 CONFÉRENCE EST — 2e Tour')
east_r2_matchups = [
    ('NYK', 'PHI', 0, 0, True),    # Knicks (dom.) vs 76ers
    ('CLE', 'DET', 0, 0, False),    # Cavaliers/Raptors vs Pistons — à ajuster
]

east_winners_r2 = []
for t1, t2, w1, w2, h in east_r2_matchups:
    winner, conf, score = predict_series(
        t1, t2, team_stats, w1, w2, h, model=MODEL, verbose=True
    )
    east_winners_r2.append((winner, conf))

# OUEST
print('\n🔴 CONFÉRENCE OUEST — 2e Tour')
west_r2_matchups = [
    ('OKC', 'LAL', 0, 0, True),    # Thunder (dom.) vs Lakers
    ('SAS', 'MIN', 0, 0, True),    # Wolves (dom.) vs Nuggets
]

west_winners_r2 = []
for t1, t2, w1, w2, h in west_r2_matchups:
    winner, conf, score = predict_series(
        t1, t2, team_stats, w1, w2, h, model=MODEL, verbose=True
    )
    west_winners_r2.append((winner, conf))


🔵 CONFÉRENCE EST — 2e Tour

🏀 SÉRIE : Knicks vs 76ers (xgboost)
   P(Knicks gagne la série) = 82.8%
   P(76ers gagne la série) = 17.2%
   Durée moyenne prédite     = 5.5 matchs
   → Knicks (82.8% de confiance)

🏀 SÉRIE : Cavaliers vs Pistons (xgboost)
   P(Cavaliers gagne la série) = 28.9%
   P(Pistons gagne la série) = 71.0%
   Durée moyenne prédite     = 5.7 matchs
   → Pistons (71.0% de confiance)

🔴 CONFÉRENCE OUEST — 2e Tour

🏀 SÉRIE : Thunder vs Lakers (xgboost)
   P(Thunder gagne la série) = 92.0%
   P(Lakers gagne la série) = 8.0%
   Durée moyenne prédite     = 5.2 matchs
   → Thunder (92.0% de confiance)

🏀 SÉRIE : Spurs vs Timberwolves (xgboost)
   P(Spurs gagne la série) = 78.0%
   P(Timberwolves gagne la série) = 22.0%
   Durée moyenne prédite     = 5.6 matchs
   → Spurs (78.0% de confiance)


In [45]:
# Finales de conférence
print('\n' + '=' * 60)
print('FINALES DE CONFÉRENCE')
print('=' * 60)

print('\n🔵 FINALE EST')
t1, c1 = east_winners_r2[0]
t2, c2 = east_winners_r2[1]
east_finalist, conf_e, score_e = predict_series(
    t1, t2, team_stats, 0, 0,
    team1_has_home=(c1 > c2),
    model=MODEL, verbose=True
)

print('\n🔴 FINALE OUEST')
t1, c1 = west_winners_r2[0]
t2, c2 = west_winners_r2[1]
west_finalist, conf_w, score_w = predict_series(
    t1, t2, team_stats, 0, 0,
    team1_has_home=(c1 > c2),
    model=MODEL, verbose=True
)


FINALES DE CONFÉRENCE

🔵 FINALE EST

🏀 SÉRIE : Knicks vs Pistons (xgboost)
   P(Knicks gagne la série) = 40.4%
   P(Pistons gagne la série) = 59.6%
   Durée moyenne prédite     = 5.8 matchs
   → Pistons (59.6% de confiance)

🔴 FINALE OUEST

🏀 SÉRIE : Thunder vs Spurs (xgboost)
   P(Thunder gagne la série) = 65.8%
   P(Spurs gagne la série) = 34.2%
   Durée moyenne prédite     = 5.7 matchs
   → Thunder (65.8% de confiance)


In [46]:
# FINALES NBA
print('\n' + '=' * 60)
print('🏆 FINALES NBA 2025-26')
print('=' * 60)

champion, conf_f, score_f = predict_series(
    east_finalist, west_finalist, team_stats,
    0, 0,
    team1_has_home=(conf_e > conf_w),
    model=MODEL, verbose=True
)

print('\n' + '=' * 60)
print(f'🏆 CHAMPION NBA PRÉDIT : {champion}')
print(f'   Modèle utilisé      : {MODEL.upper()}')
print(f'   Finale              : {east_finalist} vs {west_finalist}')
print(f'   Score prédit        : {score_f}')
print('=' * 60)


🏆 FINALES NBA 2025-26

🏀 SÉRIE : Pistons vs Thunder (xgboost)
   P(Pistons gagne la série) = 36.4%
   P(Thunder gagne la série) = 63.6%
   Durée moyenne prédite     = 5.8 matchs
   → Thunder (63.6% de confiance)

🏆 CHAMPION NBA PRÉDIT : Thunder
   Modèle utilisé      : XGBOOST
   Finale              : Pistons vs Thunder
   Score prédit        : 4-2
